In [ ]:
"""
=============================================================================
MPA-MLF Final Project - Classification of Room Occupancy
=============================================================================
Delay-Doppler 60 GHz signal classification → 4 classes (0, 1, 2, 3 persons)
Strategy: Transfer Learning with EfficientNet-B0 + Data Augmentation + Ensemble

HOW TO USE ON GOOGLE COLAB:
1. Upload x_train.zip, x_test.zip, y_train_v2.csv, y_test_submission_example_v2.csv
2. Run all cells in order
3. Download the submission CSV at the end
=============================================================================
"""

# =============================================================================
# CELL 1 — Setup & Imports
# =============================================================================

# Uncomment this line on Google Colab to install missing packages:
# !pip install timm seaborn
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import timm  # PyTorch Image Models — pour EfficientNet, ResNet, etc.

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# =============================================================================
# CELL 2 — Configuration
# =============================================================================

class Config:
    # Paths — MODIFY THESE if your files are elsewhere on Colab
    TRAIN_DIR = "C:\Users\Victoire\Documents\5-persoVictoire\BRNO-VUT\Project_ML\data\x_train"        # Folder with training images (after unzipping)
    TEST_DIR = "C:\Users\Victoire\Documents\5-persoVictoire\BRNO-VUT\Project_ML\data\x_test"          # Folder with test images (after unzipping)
    TRAIN_LABELS = "C:\Users\Victoire\Documents\5-persoVictoire\BRNO-VUT\Project_ML\data\y_train_v2.csv"
    SUBMISSION_EXAMPLE = "C:\Users\Victoire\Documents\5-persoVictoire\BRNO-VUT\Project_ML\data\y_test_submission_example_v2.csv"

    # Model
    MODEL_NAME = "efficientnet_b0"  # Options: efficientnet_b0, resnet18, resnet34, convnext_tiny
    NUM_CLASSES = 4
    IMG_SIZE = 128                  # Resize images to this (128 is good for small images)
    IN_CHANNELS = 3                 # RGB

    # Training
    BATCH_SIZE = 64
    NUM_EPOCHS = 30
    LEARNING_RATE = 1e-3            # For the classifier head
    LR_BACKBONE = 1e-4              # For the pretrained backbone (lower!)
    WEIGHT_DECAY = 1e-4
    NUM_FOLDS = 5                   # K-Fold cross-validation
    TRAIN_FOLDS = [0, 1, 2, 3]     # Which folds to train on (fold 4 = validation)

    # Data Augmentation
    USE_AUGMENTATION = True
    USE_MIXUP = True
    MIXUP_ALPHA = 0.2

    # Ensemble
    USE_ENSEMBLE = True             # Train on multiple folds and average predictions
    ENSEMBLE_FOLDS = [0, 1, 2, 3, 4]  # Train 5 models, one per fold

    # TTA (Test-Time Augmentation)
    USE_TTA = True
    TTA_ROUNDS = 5

cfg = Config()

# =============================================================================
# CELL 3 — Unzip Data (Colab)
# =============================================================================

# Uncomment these lines on Google Colab:
# !unzip -q x_train.zip -d x_train
# !unzip -q x_test.zip -d x_test

# Verify files exist
if os.path.exists(cfg.TRAIN_DIR):
    train_files = os.listdir(cfg.TRAIN_DIR)
    print(f"Training images found: {len(train_files)}")
else:
    print(f"⚠️  {cfg.TRAIN_DIR} not found — unzip x_train.zip first!")

if os.path.exists(cfg.TEST_DIR):
    test_files = os.listdir(cfg.TEST_DIR)
    print(f"Test images found: {len(test_files)}")
else:
    print(f"⚠️  {cfg.TEST_DIR} not found — unzip x_test.zip first!")

# =============================================================================
# CELL 4 — Load and Explore Labels
# =============================================================================

df_train = pd.read_csv(cfg.TRAIN_LABELS)
df_sub = pd.read_csv(cfg.SUBMISSION_EXAMPLE)

print(f"Training samples: {len(df_train)}")
print(f"Test samples:     {len(df_sub)}")
print(f"\nClass distribution:")
print(df_train['target'].value_counts().sort_index())

# NOTE IMPORTANT: Les labels sont indexés à partir de 0, les images à partir de 1
# Donc label id=0 → img_1.png, id=1 → img_2.png, etc.
# On crée la colonne filename
df_train['filename'] = df_train['id'].apply(lambda x: f"img_{x+1}.png")
df_sub['filename'] = df_sub['id'].apply(lambda x: f"img_{x+1}.png")

# Vérification
print(f"\nFirst 5 mappings:")
print(df_train[['id', 'filename', 'target']].head())

# Visualize class distribution
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
class_names = ['0 person', '1 person', '2 persons', '3 persons']
counts = df_train['target'].value_counts().sort_index()
bars = ax.bar(class_names, counts.values, color=['#2196F3', '#4CAF50', '#FF9800', '#F44336'])
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', fontsize=12)
ax.set_ylabel('Count')
ax.set_title('Class Distribution in Training Set')
plt.tight_layout()
plt.show()

# =============================================================================
# CELL 5 — Visualize Sample Images
# =============================================================================

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for class_id in range(4):
    samples = df_train[df_train['target'] == class_id].sample(2, random_state=SEED)
    for j, (_, row) in enumerate(samples.iterrows()):
        img_path = os.path.join(cfg.TRAIN_DIR, row['filename'])
        if os.path.exists(img_path):
            img = Image.open(img_path)
            axes[j, class_id].imshow(img)
        axes[j, class_id].set_title(f"Class {class_id}: {class_names[class_id]}")
        axes[j, class_id].axis('off')
plt.suptitle("Sample Images per Class", fontsize=14)
plt.tight_layout()
plt.show()

# =============================================================================
# CELL 6 — Dataset Class
# =============================================================================

class RoomOccupancyDataset(Dataset):
    """
    Custom Dataset for delay-Doppler images.
    """
    def __init__(self, dataframe, img_dir, transform=None, is_test=False):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, row['id']
        else:
            label = row['target']
            return image, label

# =============================================================================
# CELL 7 — Data Augmentation Transforms
# =============================================================================

# Training transforms — with augmentation
train_transform = T.Compose([
    T.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                std=[0.229, 0.224, 0.225]),
])

# Validation / Test transforms — NO augmentation
val_transform = T.Compose([
    T.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# TTA transforms — light augmentation for test-time
tta_transform = T.Compose([
    T.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

print("✅ Transforms defined")

# =============================================================================
# CELL 8 — Model Definition
# =============================================================================

class OccupancyClassifier(nn.Module):
    """
    Transfer Learning model using a pretrained backbone + custom head.
    """
    def __init__(self, model_name=cfg.MODEL_NAME, num_classes=cfg.NUM_CLASSES, pretrained=True):
        super().__init__()

        # Load pretrained backbone from timm
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        num_features = self.backbone.num_features

        # Custom classification head with dropout for regularization
        self.head = nn.Sequential(
            nn.BatchNorm1d(num_features),
            nn.Dropout(0.3),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

# Quick test
model_test = OccupancyClassifier()
dummy = torch.randn(2, 3, cfg.IMG_SIZE, cfg.IMG_SIZE)
out = model_test(dummy)
print(f"✅ Model output shape: {out.shape} (batch=2, classes={cfg.NUM_CLASSES})")
del model_test

# =============================================================================
# CELL 9 — Mixup Augmentation
# =============================================================================

def mixup_data(x, y, alpha=0.2):
    """
    Mixup: mélange deux exemples et leurs labels.
    Aide à la régularisation et améliore la généralisation.
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Loss function for mixup."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# =============================================================================
# CELL 10 — Training Function
# =============================================================================

def train_one_epoch(model, loader, criterion, optimizer, scheduler, epoch, use_mixup=False):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        if use_mixup and np.random.rand() > 0.5:  # 50% chance of mixup
            images, labels_a, labels_b, lam = mixup_data(images, labels, cfg.MIXUP_ALPHA)
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            # For accuracy, use original labels
            _, predicted = outputs.max(1)
            correct += (lam * predicted.eq(labels_a).sum().item()
                       + (1 - lam) * predicted.eq(labels_b).sum().item())
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()

        total += labels.size(0)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)

    if scheduler is not None:
        scheduler.step()

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
            running_loss += loss.item() * labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels)

# =============================================================================
# CELL 11 — Full Training Pipeline with K-Fold
# =============================================================================

def train_fold(df_train_data, fold_idx, n_folds=cfg.NUM_FOLDS):
    """
    Train one model on a specific fold split.
    Returns the trained model and validation metrics.
    """
    print(f"\n{'='*60}")
    print(f"  FOLD {fold_idx + 1}/{n_folds}")
    print(f"{'='*60}")

    # Split data
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    splits = list(skf.split(df_train_data, df_train_data['target']))
    train_idx, val_idx = splits[fold_idx]

    df_fold_train = df_train_data.iloc[train_idx]
    df_fold_val = df_train_data.iloc[val_idx]

    print(f"  Train: {len(df_fold_train)} | Val: {len(df_fold_val)}")

    # Weighted sampler for class imbalance
    class_counts = df_fold_train['target'].value_counts().sort_index().values
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[t] for t in df_fold_train['target'].values]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    # Datasets & Loaders
    train_dataset = RoomOccupancyDataset(df_fold_train, cfg.TRAIN_DIR, train_transform)
    val_dataset = RoomOccupancyDataset(df_fold_val, cfg.TRAIN_DIR, val_transform)

    train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE,
                              sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)

    # Model
    model = OccupancyClassifier().to(DEVICE)

    # Loss with class weights
    weight_tensor = torch.tensor(class_weights / class_weights.sum() * len(class_weights),
                                  dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weight_tensor)

    # Optimizer — different LR for backbone vs head
    optimizer = optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': cfg.LR_BACKBONE},
        {'params': model.head.parameters(), 'lr': cfg.LEARNING_RATE},
    ], weight_decay=cfg.WEIGHT_DECAY)

    # Cosine Annealing scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.NUM_EPOCHS, eta_min=1e-6)

    # Training loop
    best_val_acc = 0
    best_model_state = None
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(cfg.NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, epoch,
            use_mixup=cfg.USE_MIXUP
        )
        val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            best_preds = val_preds
            best_labels = val_labels

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{cfg.NUM_EPOCHS} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.1f}% | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.1f}% | "
                  f"Best: {best_val_acc:.1f}%")

    # Load best model
    model.load_state_dict(best_model_state)
    print(f"\n  ✅ Best validation accuracy: {best_val_acc:.2f}%")

    return model, history, best_val_acc, best_preds, best_labels

# =============================================================================
# CELL 12 — Train All Folds (Ensemble)
# =============================================================================

models = []
all_histories = []
fold_accuracies = []

if cfg.USE_ENSEMBLE:
    folds_to_train = cfg.ENSEMBLE_FOLDS
else:
    folds_to_train = [0]  # Just one fold

for fold_idx in folds_to_train:
    model, history, val_acc, val_preds, val_labels = train_fold(df_train, fold_idx)
    models.append(model)
    all_histories.append(history)
    fold_accuracies.append(val_acc)

print(f"\n{'='*60}")
print(f"  ENSEMBLE SUMMARY")
print(f"{'='*60}")
for i, acc in enumerate(fold_accuracies):
    print(f"  Fold {i+1}: {acc:.2f}%")
print(f"  Mean:   {np.mean(fold_accuracies):.2f}% ± {np.std(fold_accuracies):.2f}%")

# =============================================================================
# CELL 13 — Plot Training Curves
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
for i, hist in enumerate(all_histories):
    axes[0].plot(hist['train_loss'], alpha=0.3, color='blue')
    axes[0].plot(hist['val_loss'], alpha=0.3, color='red')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend(['Train', 'Val'])

# Accuracy curves
for i, hist in enumerate(all_histories):
    axes[1].plot(hist['train_acc'], alpha=0.3, color='blue')
    axes[1].plot(hist['val_acc'], alpha=0.3, color='red')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend(['Train', 'Val'])

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

# =============================================================================
# CELL 14 — Confusion Matrix (last fold)
# =============================================================================

cm = confusion_matrix(val_labels, val_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix (Last Fold)')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print("\nClassification Report:")
print(classification_report(val_labels, val_preds, target_names=class_names))

# =============================================================================
# CELL 15 — Prediction Function with TTA
# =============================================================================

def predict_with_tta(models, test_loader, tta_transform, n_rounds=5):
    """
    Test-Time Augmentation: run prediction multiple times with
    random augmentations and average the results.
    """
    all_probs = []

    for model in models:
        model.eval()

        # Standard prediction (no augmentation)
        model_probs = []
        with torch.no_grad():
            for images, ids in test_loader:
                images = images.to(DEVICE)
                outputs = model(images)
                probs = torch.softmax(outputs, dim=1)
                model_probs.append(probs.cpu().numpy())
        all_probs.append(np.concatenate(model_probs, axis=0))

        # TTA rounds
        if cfg.USE_TTA:
            for round_idx in range(n_rounds):
                # Recreate dataset with TTA transforms
                tta_dataset = RoomOccupancyDataset(
                    df_sub, cfg.TEST_DIR, tta_transform, is_test=True
                )
                tta_loader = DataLoader(tta_dataset, batch_size=cfg.BATCH_SIZE,
                                        shuffle=False, num_workers=2)
                round_probs = []
                with torch.no_grad():
                    for images, ids in tta_loader:
                        images = images.to(DEVICE)
                        outputs = model(images)
                        probs = torch.softmax(outputs, dim=1)
                        round_probs.append(probs.cpu().numpy())
                all_probs.append(np.concatenate(round_probs, axis=0))

    # Average all predictions
    avg_probs = np.mean(all_probs, axis=0)
    final_preds = np.argmax(avg_probs, axis=1)
    return final_preds, avg_probs

# =============================================================================
# CELL 16 — Generate Predictions on Test Set
# =============================================================================

# Test dataset
test_dataset = RoomOccupancyDataset(df_sub, cfg.TEST_DIR, val_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE,
                         shuffle=False, num_workers=2)

print("Generating predictions...")
if cfg.USE_TTA:
    print(f"  Using TTA with {cfg.TTA_ROUNDS} rounds × {len(models)} models")
    predictions, probabilities = predict_with_tta(
        models, test_loader, tta_transform, n_rounds=cfg.TTA_ROUNDS
    )
else:
    predictions, probabilities = predict_with_tta(
        models, test_loader, tta_transform, n_rounds=0
    )

print(f"  Total predictions: {len(predictions)}")
print(f"  Prediction distribution: {Counter(predictions)}")

# =============================================================================
# CELL 17 — Create Submission File
# =============================================================================

submission = df_sub[['id']].copy()
submission['target'] = predictions
submission.to_csv('submission.csv', index=False)

print("✅ Submission saved to 'submission.csv'")
print(f"\nSubmission head:")
print(submission.head(10))
print(f"\nPrediction distribution:")
print(submission['target'].value_counts().sort_index())

# Compare with expected distribution from training set
print(f"\nTraining distribution (for comparison):")
print(df_train['target'].value_counts(normalize=True).sort_index().apply(lambda x: f"{x:.1%}"))

# =============================================================================
# CELL 18 — Confidence Analysis
# =============================================================================

# Look at prediction confidence
max_probs = probabilities.max(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of prediction confidence
axes[0].hist(max_probs, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Prediction Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Prediction Confidence')
axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='50% threshold')
axes[0].legend()

# Low confidence predictions
low_conf = (max_probs < 0.5).sum()
axes[1].bar(['High (>50%)', 'Low (<50%)'],
            [len(max_probs) - low_conf, low_conf],
            color=['#4CAF50', '#F44336'])
axes[1].set_ylabel('Count')
axes[1].set_title(f'Confidence Split ({low_conf} uncertain predictions)')

plt.tight_layout()
plt.savefig('confidence_analysis.png', dpi=150)
plt.show()

print(f"\nLow confidence predictions (<50%): {low_conf}/{len(max_probs)}")

# =============================================================================
# CELL 19 — BONUS: Quick Experiment with Different Architectures
# =============================================================================

"""
💡 Want to try other architectures? Just change cfg.MODEL_NAME:

  cfg.MODEL_NAME = "resnet18"         # Lighter, faster
  cfg.MODEL_NAME = "resnet34"         # Good balance
  cfg.MODEL_NAME = "efficientnet_b0"  # Best accuracy/speed ratio (default)
  cfg.MODEL_NAME = "efficientnet_b2"  # Heavier, potentially better
  cfg.MODEL_NAME = "convnext_tiny"    # Modern architecture
  cfg.MODEL_NAME = "mobilenetv3_small_100"  # Very fast

Then re-run cells 11-17.

For the report, you can compare 2-3 architectures and show their results.
"""

print("\n" + "="*60)
print("  🎉 PIPELINE COMPLETE!")
print("="*60)
print(f"""
Next steps:
1. Download 'submission.csv' and submit to Kaggle
2. Try different architectures (see Cell 19)
3. Use the plots for your report:
   - training_curves.png
   - confusion_matrix.png
   - confidence_analysis.png
""")